In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, explode_outer, md5, concat_ws, lit, to_json
from pyspark.sql.types import StructType, ArrayType

def appiattisci_json_relazionale(df: DataFrame, chiavi_padre: list = [], nome_livello_precedente: str = "root") -> dict:
    
    tabelle_generate = {}
    chiavi_locali = list(chiavi_padre)
    
    struttura_modificata = True
    while struttura_modificata:
        struttura_modificata = False
        nuove_colonne = []
        for campo in df.schema.fields:
            if campo.name in chiavi_locali:
                nuove_colonne.append(col(campo.name))
            elif isinstance(campo.dataType, StructType):
                struttura_modificata = True
                for sotto_campo in campo.dataType.fields:
                    nuovo_nome = f"{campo.name}_{sotto_campo.name}"
                    nuove_colonne.append(col(f"{campo.name}.{sotto_campo.name}").alias(nuovo_nome))
                    
                    if ("id" in sotto_campo.name.lower()) and nuovo_nome not in chiavi_locali:
                        chiavi_locali.append(nuovo_nome)
            else:
                nuove_colonne.append(col(campo.name))
        if struttura_modificata:
            df = df.select(nuove_colonne)

    schema = df.schema
    campi_da_mantenere = []
    array_da_separare = {} 
    
    for campo in schema.fields:
        nome_campo = campo.name
        if nome_campo in chiavi_locali:
            continue
            
        if isinstance(campo.dataType, ArrayType):
            array_da_separare[nome_campo] = campo.dataType
        else:
            campi_da_mantenere.append(col(nome_campo))
            
    colonne_finali_correnti = [col(k) for k in chiavi_locali] + campi_da_mantenere
    tabelle_generate[nome_livello_precedente] = df.select(colonne_finali_correnti).dropDuplicates()

    for nome_campo, tipo_dato in array_da_separare.items():
        
        chiavi_per_figlio = list(chiavi_locali)
        ha_id_nativo = any('id' in k.lower() for k in chiavi_per_figlio)
        for campo_corrente in df.schema.fields:
            if ("id" in campo_corrente.name.lower()):
                if campo_corrente.name not in chiavi_per_figlio:
                    chiavi_per_figlio.append(campo_corrente.name)

        tipo_elemento = tipo_dato.elementType
        
        df_esploso = df.select(
            *[col(k) for k in chiavi_per_figlio], 
            explode_outer(col(nome_campo)).alias("elemento")
        )
        
        if not ha_id_nativo:
            nome_id_figlio = f"{nome_campo}_id"
            if isinstance(tipo_elemento, StructType):
                df_esploso = df_esploso.withColumn(nome_id_figlio, md5(concat_ws("_", lit(nome_campo), to_json(col("elemento")))))
            else:
                df_esploso = df_esploso.withColumn(nome_id_figlio, md5(concat_ws("_", lit(nome_campo), col("elemento").astype("string"))))
            chiavi_per_figlio.append(nome_id_figlio)

        if isinstance(tipo_elemento, StructType):
            df_figlio = df_esploso.select(*[col(k) for k in chiavi_per_figlio], "elemento.*")
            
            for f in tipo_elemento.fields:
                if ("id" in f.name.lower()) and f.name not in chiavi_per_figlio:
                    chiavi_per_figlio.append(f.name)
        else:
            df_figlio = df_esploso.select(*[col(k) for k in chiavi_per_figlio], col("elemento").alias("elemento"))
        
        df_figlio = df_figlio.dropDuplicates()
                
        sotto_tabelle = appiattisci_json_relazionale(df_figlio, chiavi_per_figlio, nome_campo)
        for sotto_nome, sotto_df in sotto_tabelle.items():
            if sotto_nome == "root":
                tabelle_generate[nome_campo] = sotto_df
            else:
                tabelle_generate[sotto_nome] = sotto_df
            
    return tabelle_generate

In [0]:
from pyspark.sql.functions import col, element_at, split, current_timestamp
from datetime import datetime
import os

CATALOGO_TARGET = "workspace_prd"
SCHEMA_TARGET = "landing"
PREFISSO_TABELLE = "factory_details"

current_time = datetime.now()
data_caricamento = current_time.strftime("%Y-%m-%d")

path_volume="/Volumes/workspace_file_loading/staging/factory_details_raw_data/"
PATH_VOLUME_ROOT_RECURSIVE=f"{path_volume}data_loading={data_caricamento}/"

try:
    tabella_controllo = f"{CATALOGO_TARGET}.{SCHEMA_TARGET}.{PREFISSO_TABELLE}_root_summaries"
    
    file_gia_caricati = set()
    if spark.catalog.tableExists(tabella_controllo):
        print(f"Estrazione file già elaborati dalla tabella {tabella_controllo}")
        df_log = spark.table(tabella_controllo).select("nome_file_origine").distinct()
        file_gia_caricati = set([row.nome_file_origine for row in df_log.collect()])
    
    all_files_df = spark.read.format("json").load(PATH_VOLUME_ROOT_RECURSIVE) \
        .select(
            element_at(split(col("_metadata.file_path"), "/"), -1).alias("nome_file_completo"), 
            col("_metadata.file_path").alias("percorso_completo")
        ) \
        .distinct()
    
    lista_totale_files = all_files_df.collect()
    
    percorsi_file_nuovi = [row.percorso_completo for row in lista_totale_files if row.nome_file_completo not in file_gia_caricati]
    
    if not percorsi_file_nuovi:
        print("nessun file nuovo trovato nell'intero storico del Volume. Il database è già aggiornato.")
    else:
        print(f"trovati {len(percorsi_file_nuovi)} file da elaborare!")
        
        df_nuovi = spark.read \
            .option("multiline", "true") \
            .json(percorsi_file_nuovi) \
            .withColumn("nome_file_origine", element_at(split(col("_metadata.file_path"), "/"), -1)) \
            .withColumn("file_loading_dtm", current_timestamp())
            
        print("esplosione ricorsiva dei nuovi file")
        
        dizionario_tabelle = appiattisci_json_relazionale(
            df_nuovi,
            chiavi_padre=["nome_file_origine", "file_loading_dtm", "id_azienda"],
            nome_livello_precedente="root_summaries"
        )
        
        print("scrittura incrementale nel catalogo Delta")
        print("-" * 60)
        
        for nome_struttura, df_tabella in dizionario_tabelle.items():
            nome_tabella_finale = f"{CATALOGO_TARGET}.{SCHEMA_TARGET}.{PREFISSO_TABELLE}_{nome_struttura}"
            
            print(f"scrittura in corso su: {nome_tabella_finale}")
            
            df_tabella.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(nome_tabella_finale)
                
            print(f"righe aggiunte a {nome_tabella_finale}: {df_tabella.count()}")

except Exception as e:
    print(f"errore durante l'elaborazione: {str(e)}")

print("-" * 60)
print("caricamento completato")

In [0]:
catalogo = "workspace_prd"
schema = "landing"

# Ottiene la lista di tutti gli oggetti nello schema
oggetti = spark.catalog.listTables(f"{catalogo}.{schema}")

for o in oggetti:
    nome_completo = f"{catalogo}.{schema}.{o.name}"
    
    # Controlliamo il tipo: cancelliamo solo Tabelle (MANAGED/EXTERNAL) e Viste
    if o.tableType in ["MANAGED", "EXTERNAL", "VIEW"]:
        spark.sql(f"DROP TABLE IF EXISTS {nome_completo}")
        spark.sql(f"DROP VIEW IF EXISTS {nome_completo}")
        print(f"Eliminata: {nome_completo} ({o.tableType})")

print("cancellazione completata")